# Seed-0 comparison: baseline vs monotonic spatial choice

This additive notebook reproduces the seed-0 preprocessing and comparison
without modifying the baseline notebook. All generated artifacts use the
`monotonic_spatial` name and live under `results/monotonic_spatial/`.

Target convention: **Y=1 means option A was selected; Y=0 means B was
selected.** Input order: **[option A, user, option B]**.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display

tf.random.set_seed(101)
np.random.seed(101)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'Chile Primer Ciclo':
    NOTEBOOK_DIR = Path('Chile Primer Ciclo')
PROJECT_ROOT = NOTEBOOK_DIR.parent
OUTPUT_DIR = NOTEBOOK_DIR / 'results' / 'monotonic_spatial'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET = 'chile'
SEED = 101
DIM_POLITICAL_SPACE = 2
MINIMUM_NUMBER_VOTES = 10
NUM_EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 0.01

import sys
sys.path.insert(0, str(PROJECT_ROOT / 'Libreria'))
from monotonic_spatial import build_monotonic_spatial_model


## Load the exact baseline architecture without duplicating it

In [ ]:
baseline_notebook_path = NOTEBOOK_DIR / 'Entrenamiento Usuarios y propuestas - semilla 0.ipynb'
baseline_notebook = json.loads(baseline_notebook_path.read_text(encoding='utf-8'))
architecture_source = next(
    ''.join(cell['source']) for cell in baseline_notebook['cells']
    if cell['cell_type'] == 'code' and 'class PoliticalCoords' in ''.join(cell['source'])
)
baseline_namespace = {
    'tf': tf,
    'Layer': tf.keras.layers.Layer,
    'Dense': tf.keras.layers.Dense,
    'Sequential': tf.keras.Sequential,
}
exec(architecture_source, baseline_namespace)
PoliticalCoords = baseline_namespace['PoliticalCoords']
get_model_original = baseline_namespace['get_model_original']
print('Loaded unchanged baseline definitions from:', baseline_notebook_path)


## Seed-0 data preparation and exact per-user split

In [ ]:
votes_path = PROJECT_ROOT / 'data' / 'data_survey_dump_chile.csv'
bots_path = PROJECT_ROOT / 'results' / 'bot_prediction' / 'chile_uuid.csv'
cycles_path = PROJECT_ROOT / 'Libreria' / 'chile_cycles.csv'
missing = [path for path in (votes_path, bots_path, cycles_path) if not path.exists()]
if missing:
    raise FileNotFoundError(
        'The baseline training inputs are not included in this checkout: ' +
        ', '.join(str(path) for path in missing)
    )

data = pd.read_csv(votes_path)
data['datetime'] = pd.to_datetime(data['datetime'].str[:19], format='%Y-%m-%d %H:%M:%S')
data = data.sort_values('datetime')
cycles = pd.read_csv(cycles_path)
cycle_ids = cycles.loc[cycles['cycle_1'] == 1, 'id'].unique()
data = data[
    data['option_a'].isin(cycle_ids) & data['option_b'].isin(cycle_ids) &
    (data['datetime'] >= pd.Timestamp('2019-10-24 11:35:05')) &
    (data['datetime'] < pd.Timestamp('2019-10-31 01:35:54'))
].copy()
data = data[data['option_a'].notna() & data['option_b'].notna()].reset_index(drop=True)
data[['option_a', 'option_b', 'selected']] = data[['option_a', 'option_b', 'selected']].astype(int)
valid_users = data.groupby('uuid')['id'].count()
data = data[data['uuid'].isin(valid_users[valid_users >= MINIMUM_NUMBER_VOTES].index)].copy()
a = data[['option_a', 'option_b', 'selected']].to_numpy()
data['option_a_sorted'] = np.minimum(a[:, 0], a[:, 1])
data['option_b_sorted'] = np.maximum(a[:, 0], a[:, 1])
data['card_id'] = ('1' + data['option_a_sorted'].astype(str).str.zfill(3) + data['option_b_sorted'].astype(str).str.zfill(3)).astype(int)
latest = data.sort_values(['id', 'datetime']).drop_duplicates(['uuid', 'card_id'], keep='last')
bots = pd.read_csv(bots_path)
df = latest.merge(bots, on='uuid')
df = df[(df['prediction'] == 0) & (df['selected'] != 0) & df['uuid'].notna()].copy()

option_set = set(df['option_a']) | set(df['option_b'])
user_set = set(df['uuid'])
optionMap = {value: [i] for i, value in enumerate(sorted(option_set))}
userMap = {value: [i] for i, value in enumerate(sorted(user_set))}
df['Y'] = (df['selected'] == df['option_a']).astype(int)
assert ((df['Y'] == 1) == (df['selected'] == df['option_a'])).all()
X_all = np.asarray([[optionMap[a][0], userMap[u][0], optionMap[b][0]] for u, a, b in df[['uuid', 'option_a', 'option_b']].to_numpy()])
Y_all = df['Y'].to_numpy()
indexed = df.reset_index(drop=True)
train_indices, validation_indices, test_indices = [], [], []
for _, group in indexed.groupby('uuid'):
    indices = group.index.to_numpy()
    train, validation, test = np.split(
        indices, [int(len(indices) * 0.81), int(len(indices) * 0.9)]
    )
    train_indices.extend(train)
    validation_indices.extend(validation)
    test_indices.extend(test)
X, Y = X_all[train_indices], Y_all[train_indices]
X_validation, Y_validation = X_all[validation_indices], Y_all[validation_indices]
X_test, Y_test = X_all[test_indices], Y_all[test_indices]
nOptions, nUsers = len(optionMap), len(userMap)
print({'train': len(Y), 'validation': len(Y_validation), 'test': len(Y_test), 'nOptions': nOptions, 'nUsers': nUsers, 'Y=1 means A': True})


## Comparable model construction and training

In [ ]:
baseline = get_model_original(dimPoliticalSpace=DIM_POLITICAL_SPACE, nOptions=nOptions, nUsers=nUsers)
experimental = build_monotonic_spatial_model(
    dimPoliticalSpace=DIM_POLITICAL_SPACE, nOptions=nOptions, nUsers=nUsers,
    initializer_stddev=0.05, initializer_seed=SEED, compile_model=False,
)
for model in (baseline, experimental):
    model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(),
        optimizer=tf.keras.optimizers.experimental.Adamax(learning_rate=LEARNING_RATE),
        metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'), tf.keras.metrics.AUC(name='auc')],
    )

def callbacks(name):
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(
            OUTPUT_DIR / f'checkpoint_monotonic_spatial_{name}.weights.h5',
            monitor='val_loss', save_best_only=True, save_weights_only=True,
        ),
    ]
histories = {}
for name, model in [('baseline', baseline), ('monotonic_spatial', experimental)]:
    histories[name] = model.fit(
        X, Y, validation_data=(X_validation, Y_validation), epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE, callbacks=callbacks(name), verbose=2,
    )


## Evaluation, structural diagnostics, coordinates, and artifacts

In [ ]:
def chosen_is_closer_rate(coordinate_layer, x, y):
    coords = coordinate_layer(x).numpy()
    za, zu, zb = np.split(coords, 3, axis=1)
    da = np.sum((za - zu) ** 2, axis=1)
    db = np.sum((zb - zu) ** 2, axis=1)
    return np.mean(np.where(y == 1, da < db, db < da))

rows = []
for name, model in [('baseline', baseline), ('monotonic_spatial', experimental)]:
    metrics = model.evaluate(X_test, Y_test, verbose=0, return_dict=True)
    row = {
        'model': name, 'test_loss': metrics['loss'], 'test_accuracy': metrics['accuracy'],
        'test_auc': metrics['auc'], 'parameter_count': model.count_params(),
        'tau': np.nan, 'swap_mean_error': np.nan, 'swap_max_error': np.nan,
        'chosen_option_is_closer_rate': chosen_is_closer_rate(
            model.layers[0] if name == 'baseline' else model.get_layer('political_coords'),
            X_test, Y_test,
        ),
    }
    if name == 'monotonic_spatial':
        p_ab = model.predict(X_test, verbose=0)
        p_ba = model.predict(X_test[:, [2, 1, 0]], verbose=0)
        swap_error = np.abs(p_ab + p_ba - 1.0)
        assert swap_error.max() < 1e-6
        row.update({
            'tau': float(model.get_layer('monotonic_spatial_choice').tau.numpy()),
            'swap_mean_error': float(swap_error.mean()),
            'swap_max_error': float(swap_error.max()),
        })
    rows.append(row)

comparison = pd.DataFrame(rows)
display(comparison)
comparison.to_csv(OUTPUT_DIR / 'metrics_monotonic_spatial.csv', index=False)

option_index = pd.DataFrame([(key, value[0]) for key, value in optionMap.items()], columns=['option_id', 'index'])
user_index = pd.DataFrame([(key, value[0]) for key, value in userMap.items()], columns=['uuid', 'index'])
coords_p = experimental.layers[1].returnOptionsEmbedding().numpy()
coords_u = experimental.layers[1].returnUsersEmbedding().numpy()
coords_p = option_index.merge(pd.DataFrame(coords_p, columns=[f'z{i+1}' for i in range(DIM_POLITICAL_SPACE)]), left_on='index', right_index=True).drop(columns='index')
coords_u = user_index.merge(pd.DataFrame(coords_u, columns=[f'z{i+1}' for i in range(DIM_POLITICAL_SPACE)]), left_on='index', right_index=True).drop(columns='index')
display(coords_p.head())
display(coords_u.head())
coords_p.to_csv(OUTPUT_DIR / 'coordinates_monotonic_spatial_proposals.csv', index=False)
coords_u.to_csv(OUTPUT_DIR / 'coordinates_monotonic_spatial_users.csv', index=False)
model_path = OUTPUT_DIR / 'dynamap_monotonic_spatial.keras'
experimental.save(model_path)
restored = tf.keras.models.load_model(model_path)
np.testing.assert_allclose(
    restored.predict(X_test[:64], verbose=0),
    experimental.predict(X_test[:64], verbose=0),
    atol=1e-7, rtol=0.0,
)
print('Saved and reloaded:', model_path)
